In [1]:
# Importing Modules
import pandas as pd
import itertools
from sklearn.model_selection import train_test_split
from src.dataloader import loadData
from src.model import GNNModel
from src.train_test import TrainGNN, TestGNN
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error

In [2]:
# Loading ESOL data
esol_data = pd.read_csv("data/processed/train/ESOL.csv")

# Loading Lipophilicity data
lipophilicity_data = pd.read_csv("data/processed/train/Lipophilicity.csv")

# Loading B3DB data
b3db_data = pd.read_csv("data/processed/train/B3DB.csv")

# Loading RT data
rt_data = pd.read_csv("data/processed/train/RT.csv")

In [3]:
# Function to run analysis
def RunGNN(data, dataName, modelName, h_dim, b_size, lr, d_out, wd):
    
	# SMILES (To be used to generate molecular graph, training and testing)
	smiles_X = data["smiles"]
	X = smiles_X.to_numpy()

	# Target labels (log solubility values)
	y = data["target"]
	y = y.to_numpy()

	# Train test split
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

	# Loading dataset
	train_loader = loadData(X_train, y_train, batch_size=b_size)
	test_loader = loadData(X_test, y_test, batch_size=b_size)

	# Model
	model = GNNModel(num_features=6, hidden_dim=h_dim, model_type=modelName, dropout=d_out)

	# Model training
	model = TrainGNN(model, train_loader, epochs=100, learning_rate=lr, w_decay=wd)

	# Model testing
	y_test, y_pred = TestGNN(model, test_loader)

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)

	# Return results
	return pd.DataFrame({"Data":[dataName], "Model":[modelName],
            "h_dim":[h_dim], "b_size":[b_size], "lr":[lr],
            "w_decay":[wd], "d_out":[d_out],
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)]})

In [4]:
# Function for model selection and hyperparameter tuning
def search_hyperparams(dataName):
    temp_out = []
    # Hyperparameter tuning loop
    for modelName in models:
        for i in grid_search_list:
            temp_out.append(RunGNN(data_dict[dataName], dataName, modelName, i["h_dim"], i["b_size"], i["lr"], i["dropout"], i["weight_decay"]))

    # Saving results
    final_df = pd.concat(temp_out, ignore_index=True)
    final_df.to_csv(f"results/Output_Hyperparameter_Optimization_GNN_{dataName}.csv", quoting=False, index=False)

    # Best parameters
    for model in models:
        temp = final_df[final_df["Model"]==model]
        print(temp.sort_values(by=["RMSE"]).head(1),"\n")

In [5]:
# Models
models = ["GCN", "GAT", "GIN", "GraphSAGE"]

# Hyperparameter dict
hyperparams = {
    'lr': [0.001, 0.0005],
    'h_dim': [16, 32, 64],
    'b_size': [16, 32],
    'weight_decay': [1e-4, 1e-5], 
    'dropout': [0.2, 0.5] 
}

# Data
data_dict = {
    "ESOL": esol_data, 
    "Lipophilicity": lipophilicity_data,
    "RT": rt_data,
    "B3DB": b3db_data
}

# Grid search list
keys = hyperparams.keys()
values = hyperparams.values()
grid_search_list = [dict(zip(keys, combination)) for combination in itertools.product(*values)]

In [6]:
# Params Optim for ESOL
search_hyperparams("ESOL")

    Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
16  ESOL   GCN     64      16  0.001   0.0001    0.2  1.05   1.4 

    Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
68  ESOL   GAT     64      32  0.001   0.0001    0.2  1.02  1.31 

     Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
116  ESOL   GIN     64      32  0.001   0.0001    0.2  1.04  1.32 

     Data      Model  h_dim  b_size     lr  w_decay  d_out  MAE  RMSE
161  ESOL  GraphSAGE     64      16  0.001   0.0001    0.5  1.0  1.35 



In [7]:
# Params Optim for Lipophilicity
search_hyperparams("Lipophilicity")

             Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
10  Lipophilicity   GCN     32      16  0.001  0.00001    0.2  0.99  1.21 

             Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
68  Lipophilicity   GAT     64      32  0.001   0.0001    0.2  0.98  1.19 

              Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
118  Lipophilicity   GIN     64      32  0.001  0.00001    0.2  0.97  1.19 

              Data      Model  h_dim  b_size     lr  w_decay  d_out   MAE  \
161  Lipophilicity  GraphSAGE     64      16  0.001   0.0001    0.5  0.98   

     RMSE  
161  1.19   



In [8]:
# Params Optim for RT
search_hyperparams("RT")

   Data Model  h_dim  b_size     lr  w_decay  d_out     MAE   RMSE
18   RT   GCN     64      16  0.001  0.00001    0.2  127.44  150.1 

   Data Model  h_dim  b_size     lr  w_decay  d_out     MAE    RMSE
64   RT   GAT     64      16  0.001   0.0001    0.2  120.77  142.67 

    Data Model  h_dim  b_size     lr  w_decay  d_out     MAE    RMSE
113   RT   GIN     64      16  0.001   0.0001    0.5  114.83  139.41 

    Data      Model  h_dim  b_size     lr  w_decay  d_out     MAE    RMSE
162   RT  GraphSAGE     64      16  0.001  0.00001    0.2  126.39  149.03 



In [9]:
# Params Optim for B3DB
search_hyperparams("B3DB")

    Data Model  h_dim  b_size      lr  w_decay  d_out  MAE  RMSE
40  B3DB   GCN     64      16  0.0005   0.0001    0.2  0.5  0.62 

    Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
58  B3DB   GAT     32      16  0.001  0.00001    0.2  0.46  0.58 

     Data Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
116  B3DB   GIN     64      32  0.001   0.0001    0.2  0.47  0.58 

     Data      Model  h_dim  b_size     lr  w_decay  d_out   MAE  RMSE
166  B3DB  GraphSAGE     64      32  0.001  0.00001    0.2  0.46  0.58 

